# VeriPix - Phase 1 Results Notebook (FatFormer baseline)
Runs the **official FatFormer** code on a **small test subset** and produces every table and figure for the Phase 1 slides.

**Kaggle settings:** Accelerator = GPU (T4 / P100), Internet = ON. Run cells top to bottom.

**Before you start (Cell 5):** add your data as a Kaggle Dataset with this layout (the FatFormer layout):
```
<dataset>/test/<generator>/0_real/*.png   (real images)
<dataset>/test/<generator>/1_fake/*.png   (fake images)
```
Use 50-200 images per class per generator. Generator folder names: progan, stylegan, stylegan2, biggan, cyclegan, stargan, gaugan, deepfake, dalle, glide_50_27, glide_100_10, glide_100_27, guided, ldm_100, ldm_200, ldm_200_cfg.

**Honesty rules for the slides:** these numbers come from a *small subset*, so they will differ from the paper's full-test-set numbers. Label them "reproduced on a mini subset". Never present the paper's numbers as yours.

## 1. Environment (screenshot for the feasibility slide)

In [ ]:
import os, sys, time, json
import torch
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory/1e9, 1), "GB")
else:
    print("WARNING: no GPU. Turn on Settings > Accelerator > GPU.")

## 2. Get the official code and install the extra library

In [ ]:
!git clone https://github.com/Michel-liu/FatFormer.git
%cd FatFormer
!pip install -q pytorch_wavelets PyWavelets gdown ftfy regex

In [ ]:
# Small compatibility fix: newer setuptools removed 'packaging' from pkg_resources
p = "models/clip/clip.py"
t = open(p).read()
t = t.replace("from pkg_resources import packaging", "import packaging, packaging.version")
open(p, "w").write(t)
print("patched", p)

## 3. Download the CLIP ViT-L/14 weights and the FatFormer checkpoint
Links are the ones given in the official README. If the Google Drive download fails (quota), download `fatformer` checkpoint by hand and add it as a Kaggle Dataset, then set `CKPT_PATH` below.

In [ ]:
!mkdir -p pretrained ckpt
!wget -q -O pretrained/ViT-L-14.pt https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt
!gdown "https://drive.google.com/uc?id=1Q_Kgq4ygDf8XEHgAf-SgDN6Ru_IOTLkj" -O ckpt/fatformer.pth
!ls -lh pretrained ckpt

In [ ]:
CKPT_PATH = "ckpt/fatformer.pth"      # change if you uploaded the checkpoint as a Kaggle Dataset
assert os.path.getsize(CKPT_PATH) > 1e6, "Checkpoint missing or too small - see the note above"
assert os.path.getsize("pretrained/ViT-L-14.pt") > 5e8, "CLIP weights missing - re-run the wget cell"
print("files OK")

## 4. Find your dataset

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/")

## 5. Set the dataset path and generators
`DATA_ROOT` is the folder that **contains** the `test/` folder.

In [ ]:
DATA_ROOT = "/kaggle/input/YOUR-DATASET-NAME"      # <-- edit this
TEST_DIR = os.path.join(DATA_ROOT, "test")

GAN_NAMES = ["progan","stylegan","stylegan2","biggan","cyclegan","stargan","gaugan","deepfake"]
DM_NAMES  = ["dalle","glide_50_27","glide_100_10","glide_100_27","guided","ldm_100","ldm_200","ldm_200_cfg","PNDM"]
available = sorted(os.listdir(TEST_DIR))
GENS = [g for g in GAN_NAMES + DM_NAMES if g in available]
print("Found generators:", GENS)
print("Ignored folders :", [a for a in available if a not in GENS])

def group_of(g):
    if g == "progan": return "seen"
    return "unseen GAN" if g in GAN_NAMES else "unseen diffusion"

### (Optional) Make a small subset from a big dataset
Only use this if you attached a big dataset. It links N images per class instead of copying them.

In [ ]:
import random, glob
def make_subset(src_test, dst_test, n_per_class=100, seed=42):
    random.seed(seed)
    for dp, dn, fn in os.walk(src_test):
        if os.path.basename(dp) in ("0_real", "1_fake"):
            files = sorted(f for f in fn if f.lower().endswith((".png",".jpg",".jpeg",".webp",".bmp")))
            pick = random.sample(files, min(n_per_class, len(files)))
            out = os.path.join(dst_test, os.path.relpath(dp, src_test))
            os.makedirs(out, exist_ok=True)
            for f in pick:
                d = os.path.join(out, f)
                if not os.path.exists(d): os.symlink(os.path.join(dp, f), d)
    print("subset ready at", dst_test)
# make_subset("/kaggle/input/BIG-DATASET/test", "/kaggle/working/mini/test", 100)
# then set DATA_ROOT = "/kaggle/working/mini" and TEST_DIR above

## 6. Build the model and load the checkpoint

In [ ]:
sys.path.insert(0, os.getcwd())
import numpy as np
from main import get_args_parser
from models import build_model

args = get_args_parser().parse_args(["--test_selected_subsets", "progan",
                                     "--num_vit_adapter", "3", "--num_context_embedding", "8"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(args)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(model.load_state_dict(ck["model"]))
model = model.to(device).eval()
n_all = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_all/1e6:.1f} M")

## 7. Run inference on every generator

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from utils.dataset import Dataset_Creator

creator = Dataset_Creator(dataset_path=DATA_ROOT, batch_size=32, num_workers=2,
                          img_resolution=256, crop_resolution=224)
datasets, names = creator.build_dataset("test", selected_subsets=GENS)

def paths_of(ds):
    if hasattr(ds, "datasets"):
        return [p for d in ds.datasets for p, _ in d.samples]
    return [p for p, _ in ds.samples]

@torch.no_grad()
def run(ds):
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    ys, ps, t_total = [], [], 0.0
    for x, y in tqdm(dl, leave=False):
        x = x.to(device)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        out = model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t_total += time.time() - t0
        ps += out.softmax(dim=1)[:, 1].cpu().tolist()
        ys += y.tolist()
    return np.array(ys), np.array(ps), 1000 * t_total / max(len(ys), 1)

RESULTS = {}
for name, ds in zip(names, datasets):
    y, p, ms = run(ds)
    RESULTS[name] = dict(y=y, p=p, ms=ms, paths=paths_of(ds))
    print(f"{name:14s} images={len(y):5d}  {ms:.1f} ms/image")

## 8. Metrics table (goes on the Results slide)

In [ ]:
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)
RES = "/kaggle/working/results"; os.makedirs(RES, exist_ok=True)

def metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(set(y.tolist())) > 1
    return dict(n_real=int((y == 0).sum()), n_fake=int((y == 1).sum()),
        acc=100*accuracy_score(y, pred),
        real_acc=100*(pred[y == 0] == 0).mean() if (y == 0).any() else np.nan,
        fake_acc=100*(pred[y == 1] == 1).mean() if (y == 1).any() else np.nan,
        precision=100*precision_score(y, pred, zero_division=0),
        recall=100*recall_score(y, pred, zero_division=0),
        f1=100*f1_score(y, pred, zero_division=0),
        ap=100*average_precision_score(y, p) if both else np.nan,
        auc=100*roc_auc_score(y, p) if both else np.nan,
        TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))

rows = []
for g, r in RESULTS.items():
    m = metrics(r["y"], r["p"]); m.update(generator=g, group=group_of(g), ms_per_image=r["ms"]); rows.append(m)
df = pd.DataFrame(rows)
cols = ["generator","group","n_real","n_fake","acc","real_acc","fake_acc","precision","recall","f1","ap","auc","TN","FP","FN","TP","ms_per_image"]
df = df[cols]
mean = df[["acc","real_acc","fake_acc","precision","recall","f1","ap","auc","ms_per_image"]].mean()
df_show = pd.concat([df, pd.DataFrame([dict(generator="MEAN", group="-", **mean.to_dict())])], ignore_index=True)
df_show.round(1).to_csv(f"{RES}/results_table.csv", index=False)
display(df_show.round(1))

grp = df.groupby("group")[["acc","ap","f1"]].mean().round(1)
display(grp); grp.to_csv(f"{RES}/group_summary.csv")

## 9. Dataset table with verified counts (replace 'To be verified' on the Dataset slide)

In [ ]:
dtab = df[["generator","group","n_real","n_fake"]].copy()
dtab["total"] = dtab.n_real + dtab.n_fake
display(dtab); dtab.to_csv(f"{RES}/dataset_counts.csv", index=False)
print("Total images used:", int(dtab.total.sum()))

## 10. Comparison with the paper (clearly labelled)
Paper numbers are from the official README (4-class ProGAN training, **full** test sets). Check which checkpoint you downloaded before quoting.

In [ ]:
PAPER = {"progan":(99.9,100.0),"stylegan":(97.2,99.8),"stylegan2":(98.8,100.0),"biggan":(99.5,100.0),
 "cyclegan":(99.3,100.0),"stargan":(99.8,100.0),"gaugan":(99.4,100.0),"deepfake":(93.2,98.0),
 "dalle":(98.8,99.8),"glide_100_27":(94.4,99.1),"glide_50_27":(94.7,99.4),"glide_100_10":(94.2,99.2),
 "guided":(76.1,92.0),"ldm_200":(98.6,99.8),"ldm_200_cfg":(94.9,99.1),"ldm_100":(98.7,99.9),"PNDM":(99.3,100.0)}
cmp = df[["generator","acc","ap"]].copy()
cmp["paper_acc"] = cmp.generator.map(lambda g: PAPER.get(g, (np.nan, np.nan))[0])
cmp["paper_ap"]  = cmp.generator.map(lambda g: PAPER.get(g, (np.nan, np.nan))[1])
cmp["acc_diff"] = cmp.acc - cmp.paper_acc
cmp = cmp.rename(columns={"acc":"our_acc (mini subset)","ap":"our_ap (mini subset)"})
display(cmp.round(1)); cmp.round(1).to_csv(f"{RES}/comparison_with_paper.csv", index=False)

## 11. Figures for the slides (saved as PNG in /kaggle/working/results)

In [ ]:
import matplotlib.pyplot as plt
COL = {"seen":"#0E8C8C", "unseen GAN":"#12284C", "unseen diffusion":"#B45309"}
plt.rcParams.update({"font.family":"serif", "font.size":11})

def bar(col, title, fn):
    d = df
    plt.figure(figsize=(9, 4))
    plt.bar(d.generator, d[col], color=[COL[g] for g in d.group])
    plt.ylim(0, 105); plt.xticks(rotation=45, ha="right"); plt.ylabel(col + " (%)"); plt.title(title)
    for g, c in COL.items(): plt.bar(0, 0, color=c, label=g)
    plt.legend(loc="lower left"); plt.tight_layout(); plt.savefig(f"{RES}/{fn}", dpi=200); plt.show()
bar("acc", "Accuracy per generator (mini subset)", "fig_accuracy.png")
bar("ap", "Average precision per generator (mini subset)", "fig_ap.png")

In [ ]:
# Confusion matrix: all generators pooled
from sklearn.metrics import roc_curve
Y = np.concatenate([r["y"] for r in RESULTS.values()]); Pp = np.concatenate([r["p"] for r in RESULTS.values()])
def cm_plot(y, p, title, fn):
    cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
    plt.figure(figsize=(4, 3.6)); plt.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14, color="black")
    plt.xticks([0, 1], ["Real", "Fake"]); plt.yticks([0, 1], ["Real", "Fake"])
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(title); plt.tight_layout()
    plt.savefig(f"{RES}/{fn}", dpi=200); plt.show()
cm_plot(Y, Pp, "Confusion matrix (all generators)", "fig_confusion_all.png")
unseen = [g for g in RESULTS if group_of(g) != "seen"]
if unseen:
    cm_plot(np.concatenate([RESULTS[g]["y"] for g in unseen]), np.concatenate([RESULTS[g]["p"] for g in unseen]),
            "Confusion matrix (unseen generators)", "fig_confusion_unseen.png")

In [ ]:
# ROC curves
plt.figure(figsize=(6, 5))
for g, r in RESULTS.items():
    if len(set(r["y"].tolist())) > 1:
        fpr, tpr, _ = roc_curve(r["y"], r["p"]); plt.plot(fpr, tpr, label=g)
plt.plot([0, 1], [0, 1], "k--", lw=1); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves per generator"); plt.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig(f"{RES}/fig_roc.png", dpi=200); plt.show()

# Score distributions (like Fig. 2 of the paper)
show = (["progan"] if "progan" in RESULTS else []) + [g for g in RESULTS if g != "progan"][:3]
fig, ax = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3), squeeze=False)
for a, g in zip(ax[0], show):
    r = RESULTS[g]
    a.hist(r["p"][r["y"] == 0], bins=20, range=(0, 1), alpha=0.7, label="real", color="#2563EB")
    a.hist(r["p"][r["y"] == 1], bins=20, range=(0, 1), alpha=0.7, label="fake", color="#F97316")
    a.set_title(g); a.set_xlabel("P(fake)")
ax[0][0].legend(); plt.tight_layout(); plt.savefig(f"{RES}/fig_score_distribution.png", dpi=200); plt.show()

In [ ]:
# Sample predictions grid: 4 real + 4 fake from one generator
from PIL import Image
g0 = unseen[0] if unseen else list(RESULTS)[0]
r = RESULTS[g0]
idx_r = [i for i in range(len(r["y"])) if r["y"][i] == 0][:4]
idx_f = [i for i in range(len(r["y"])) if r["y"][i] == 1][:4]
fig, ax = plt.subplots(2, 4, figsize=(12, 6.4))
for a, i in zip(ax.ravel(), idx_r + idx_f):
    a.imshow(Image.open(r["paths"][i]).convert("RGB")); a.axis("off")
    pf = r["p"][i]; true = "fake" if r["y"][i] == 1 else "real"; pred = "fake" if pf >= 0.5 else "real"
    a.set_title(f"true: {true} | pred: {pred} ({pf:.2f})", fontsize=10, color="green" if true == pred else "red")
plt.suptitle(f"Sample predictions - {g0}"); plt.tight_layout()
plt.savefig(f"{RES}/fig_samples.png", dpi=200); plt.show()

## 12. Save per-image predictions and zip everything for download

In [ ]:
rows = []
for g, r in RESULTS.items():
    for pth, y, p in zip(r["paths"], r["y"], r["p"]):
        rows.append(dict(generator=g, path=pth, label=int(y), p_fake=float(p), pred=int(p >= 0.5)))
pd.DataFrame(rows).to_csv(f"{RES}/predictions.csv", index=False)
info = dict(torch=torch.__version__, cuda=torch.cuda.is_available(),
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            params_million=round(n_all / 1e6, 1), checkpoint=CKPT_PATH, threshold=0.5)
json.dump(info, open(f"{RES}/run_info.json", "w"), indent=2)
!cd /kaggle/working && zip -qr veripix_results.zip results && ls -lh veripix_results.zip

## 13. Summary sentences for the slide (edit if needed)

In [ ]:
g = df.groupby("group")
for name in ["seen", "unseen GAN", "unseen diffusion"]:
    if name in g.groups:
        sub = g.get_group(name)
        print(f"{name:17s}: mean accuracy {sub.acc.mean():.1f}%, mean AP {sub.ap.mean():.1f}%  ({len(sub)} generators)")
print(f"Speed: {df.ms_per_image.mean():.1f} ms per image on {info['gpu']}")
print("Weakest generator by accuracy:", df.sort_values('acc').iloc[0].generator, f"({df.acc.min():.1f}%)")

## Which file goes on which slide
| File | Slide |
|---|---|
| Cell 1 screenshot + `run_info.json` | Project Status / feasibility |
| `dataset_counts.csv` | Dataset (replace "To be verified") |
| `results_table.csv` + `fig_accuracy.png` | New slide: Baseline Results (mini subset) |
| `group_summary.csv` | Seen vs unseen summary |
| `fig_confusion_all.png`, `fig_roc.png` | Evaluation / results |
| `fig_score_distribution.png` | Generalization analysis |
| `fig_samples.png` | Sample predictions |
| `comparison_with_paper.csv` | Comparison (label: paper vs mini subset) |

Caption every result as **"FatFormer released checkpoint, evaluated by us on a mini subset"**.